In [3]:
import nugget
from nugget.utils import jax_bridge


In [ ]:
device = 'cuda:0' #specify device if you want to use GPU
jax_bridge.configure_jax(platform=device) #configure jax to use GPU

<module 'jax' from '/ptmp/mpp/kristiantcho/nugget_env/lib/python3.12/site-packages/jax/__init__.py'>

In [5]:
lightsabre = nugget.surrogates.LightSabre.LightSabre(
                device=device,
                use_poisson=False, 
                particle_mode='track',
                )
signal_sampler = nugget.samplers.cyl_sampler.CylinderSampler(
    device=device,
    event_type='signal', 
    domain_size=1200, 
    E_min=1e2, 
    E_max=1e8, 
    energy_dist='log_uniform', 
    # energy_dist='power_law',
    find_exact_intersection=False,
    random_position_along_ray=True,
    uniform_zenith_sampling=True,
    cylinder_radius=600,
    cylinder_height=1000,
    cylinder_center=[0, 0, 0],
    # cos_range='vertical'
)

In [6]:
geometry = nugget.geometries.DynamicString.DynamicString(
                device=device,
                hex_type='hexagonal',
                # random_xy=True,
                domain_size=3000,  # Size of detector domain
                dim=3,  # 3D geometry
                n_strings=70,  # Initial number of detector strings
                points_per_string=20,  # Number of PMTs/sensors per string
                # custom_string_spacing=50  # Custom spacing between strings
                custom_z_spacing=50,
            )

visualizer = nugget.utils.vis_tools.Visualizer(
    device=device,
    dim=3, 
    domain_size=1400,
    # gif_temp_dir='./gif_temp_3',  # Directory for storing animation frames
    )

In [7]:
optimizer = nugget.utils.hybrid_optimizer.Optimizer(
    device=geometry.device, 
    geometry=geometry,
    visualizer=visualizer,
    # conflict_free=False,  # Single-objective optimization (vs multi-objective)
    use_alm=True,  # Use Augmented Lagrangian Method for constraints
    sigmoid_losses=True,
)

optimizer.init_geometry(
    # opt_list=[('string_weights', 0.5)],  # Learning rate for string weights (without sigmoid applied)
    opt_list=[('string_xy', 5)],  # Learning rates for string xy positions
)

Optimizing string_xy with torch.Size([70, 2]) shape


In [8]:
# Local String Repulsion: Prevents strings from getting too close to each other
local_string_repulsion_penalty = nugget.losses.geometry_penalties.LocalStringRepulsionPenalty(
    device=geometry.device
)

# String Boundary Penalty: Keeps strings within the detector domain
string_boundary_penalty = nugget.losses.geometry_penalties.StringBoundaryPenaltyCircle(
    device=geometry.device,
)

# String Number Penalty: Penalizes having too many active strings
string_number_penalty = nugget.losses.geometry_penalties.StringNumberPenalty(
    device=geometry.device
)

# Weight Binarization: Encourages string weights to be either 0 or 1
weighted_binarization_penalty = nugget.losses.geometry_penalties.WeightBinarizationPenalty(
    device=geometry.device
)

# === RESOLUTION-BASED LOSSES ===
# Angular Resolution: Optimizes detector's ability to reconstruct particle direction
weighted_angular_resolution_loss = nugget.losses.fisher_info.WeightedResolutionLoss(
    device=geometry.device,
    resolution_type='angular',
    fisher_info_params=['direction', 'position']
)

# Energy Resolution: Optimizes detector's ability to reconstruct particle energy
weighted_energy_resolution_loss = nugget.losses.fisher_info.WeightedResolutionLoss(
    device=geometry.device,
    resolution_type='energy',
    fisher_info_params=['energy']
)

jax_angular_resolution_loss = nugget.losses.fisher_info_jax.JaxResolutionLoss(
    fisher_info_params=('direction', 'position'),   # matches your res_test angular setup
    resolution_type='angular',
    lightsabre=lightsabre,        # mirrors the torch instance
)


# ROV Penalty: Accounts for physical constraints from Remotely Operated Vehicle
rov_penalty = nugget.losses.geometry_penalties.ROVPenalty(
    device=geometry.device, 
    rov_rec_width=230,  # ROV dimensions
    rov_height=160, 
    rov_tri_length=160
)

In [11]:
loss_func_dict = {
    'angular_resolution_loss': jax_angular_resolution_loss,          # JAX
    'local_string_repulsion_penalty': local_string_repulsion_penalty, # torch
    'rov_penalty': rov_penalty,                                       # torch
}

loss_params = {
    'signal_sampler': signal_sampler,
    'num_events': 1000,  # Number of events to sample per optimization step
    # 'boundary_range': 1200,  # Size of boundary region
    # 'skip_zero_response': False,
    'use_relative_energy': True,
    # 'eva_min_num_strings': 72,  # Minimum number of active strings
    'max_radius': 80,  # Maximum radius for string placement
    'num_angles': 360,  # Number of angles (divided into 360 degrees) to test for rov
    'rov_alt_mode': True,  # Whether to use alternative mode for rov penalty (see rov_penalty.py for details)
    'local_sharpness': 10,  # Sharpness parameter for local string repulsion
    # 'boundary_sharpness': 10,  # Sharpness parameter for boundary penalty
    # 'string_number_beta':5,
    # 'detach_other_probs':False,
    'rov_soft_inside':True,
    'rov_inside_sharpness': 5,
    'rov_angle_softmin_tau': 0.05,
    'rov_angle_chunk_size': 360,
    'rov_inside_use_softplus': True,
    'fisher_res_metric': 'mean',  # 'fom' 'median' 'mean'
    'constraints_list': [ 
                'rov_penalty', 
                # 'string_boundary_penalty', 
                'local_string_repulsion_penalty', 
                'string_number_penalty', 
                # 'string_weights_penalty',
                'weight_binarization_penalty',
                ],  # Constraints to enforce with ALM
}

loss_weights_dict = {
    'angular_resolution_loss': 1,
    'string_boundary_penalty': 2,  # Very high: hard constraint
    'local_string_repulsion_penalty': 0.1,
    'string_weights_penalty': 0.05,     # Encourage sparse solutions
    'string_number_penalty': 10,      # Limit detector complexity
    'weight_binarization_penalty': 0.1,
    'rov_penalty': 1,
}

loss_sigmoid_list = [
    'angular_resolution_loss',
    # 'energy_resolution_loss',
    'string_boundary_penalty',  
    'local_string_repulsion_penalty',
    'string_weights_penalty',     
    'string_number_penalty',      
    'weight_binarization_penalty',
    'rov_penalty',
]


In [10]:
plot_types = [
    'loss_components',           # Loss function values over time
    # 'uw_loss_components',        # Unweighted loss components
    # 'string_weights_scatter',    # String weight visualization
    # 'string_xy',
    'angular_resolution',
    # 'energy_resolution',
    'string_xy_rov_penalty',
    'string_xy_local_string_repulsion_penalty',
    'string_history',
    # 'nn_distance_history',
    # 'alm_mu',
    'alm_lambda',
]

vis_kwargs = {
    # 'moving_average_window': 10,  # Window size for smoothing loss curves
    # 'moving_average_losses': ['angular_resolution_loss', 'rov_penalty'],  # Losses to smooth
    'plot_types': plot_types,
    'gif_plot_selection': plot_types,  # Which plots to include in animation
    # 'loss_filter': ['string_boundary_penalty'],  # Hide large loss components from loss component plot
    # 'zoom_range':800,
    # 'max_radius': loss_params['boundary_range']/2,
    'rov_penalty_func': rov_penalty,
    'use_relative_energy': True,
    'plot_geom_contour_only': True,
    'resolution_logy_angular': True,
    'rov_draw_safe_space_union': True,
    'rov_union_per_space_colors':True,
    'rov_draw_safe_space_on_violations':False,
    # 'rov_safe_space_one_fold_only': True,
    'resolution_stat': 'mean',  # Statistic to plot for resolution ('median', 'mean', or 'fom')
}

In [12]:
# optimizer.visualizer.cleanup_gif_temp_files()
geom_dict = optimizer.optimize(
    clear_cuda_cache=True,                 # Clear GPU memory cache between iterations
    loss_func_dict=loss_func_dict,          # Dictionary of loss functions to use
    loss_weights_dict=loss_weights_dict,    # Weights for combining multiple losses
    loss_params_dict=loss_params,           # Parameters for loss function computation
    n_iter=2000,                           # Maximum number of optimization iterations
    vis_kwargs=vis_kwargs,                 # Visualization parameters
    print_freq=5,                          # Print progress every N iterations
    vis_freq=10,                           # Create plots every N iterations
    # gif_freq=5,                           # Save animation frames every N iterations
    sigmoid_loss_list=loss_sigmoid_list,         # Which losses to apply sigmoid to (for better optimization dynamics)
    revert_on_nan=True,                      # Revert geometry if loss becomes NaN
    max_nan_retries=10,                          # Maximum number of consecutive NaN reverts before stopping
    # save_geom_folder='./800main_full_hex_r600_50_u_1_fom_no_rov_patd',  # Directory to save geometry checkpoints
    # save_geom_freq=100,                      # Save geometry every N iterations
    # continue_saving=False,
    # save_best_geom_file = './best_geom_1_event.pkl',  # File to save best geometry found
    # save_last_geom = False, 
)

TypeError: 'JaxResolutionLoss' object is not callable